<a href="https://colab.research.google.com/github/naifahamanda/Studi_Kasus_Informatics_Practitioner_Lecture_Series_2026/blob/main/studi_kasus_dan_implementasi_materi_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Studi Kasus: Perancangan Dashboard Interaktif Analisis Kinerja Penjualan & Evaluasi Usabilitas (HCI / IMK)**

# 1. Latar Belakang Masalah
Sebuah perusahaan membutuhkan alat analitik bisnis (dashboard) untuk memantau performa penjualan produk secara real-time. Mengacu pada materi webinar:

Visualisasi Data yang Efektif: Grafik yang dibuat tidak boleh membingungkan atau dipenuhi chartjunk (ornamen visual berlebihan). Grafik harus memaksimalkan data-ink ratio (Edward Tufte) serta menerapkan Shneiderman's Information Seeking Mantra: "Overview first, zoom and filter, then details-on-demand".

Aspek Kognitif & Psikologi Gestalt: Antarmuka memanfaatkan preattentive attributes (warna kontras/penanda status) serta hukum Gestalt (proximity dan similarity) agar otak pengguna cepat memproses informasi tanpa kelebihan beban kognitif (cognitive load).

Evaluasi Usabilitas (HCI Evaluation): Tingkat kemudahan antarmuka dievaluasi menggunakan System Usability Scale (SUS) dan indikator Heuristic Evaluation Jakob Nielsen.

# 2. Alur Kerja (Workflow IMK & Visualisasi Data)
Generasi Data & Preprocessing: Menyiapkan dataset transaksi penjualan.

Pembuatan Dashboard Interaktif: Menggunakan library Python Plotly & Dash / ipywidgets untuk menciptakan visualisasi data yang menerapkan prinsip overview, filter, dan details-on-demand.

Simulasi Evaluasi Usabilitas (SUS Analysis): Menghitung skor kemudahan antarmuka berdasarkan respon kuesioner pengguna.

In [ ]:
# Install library interaktif jika belum terpasang
!pip install -q plotly ipywidgets pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

print("✅ Library IMK & Visualisasi Data berhasil dimuat!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.1 MB/s eta 0:00:00
✅ Library IMK & Visualisasi Data berhasil dimuat!


In [ ]:
np.random.seed(42)
n_samples = 300

dates = pd.date_range(start='2026-01-01', periods=n_samples, freq='D')
regions = ['Bengkulu', 'Jakarta', 'Surabaya', 'Medan']
categories = ['Elektronik', 'Pakaian', ' Makanan & Minuman', 'Otomotif']

data = {
    'Tanggal': np.random.choice(dates, size=n_samples),
    'Wilayah': np.random.choice(regions, size=n_samples, p=[0.4, 0.3, 0.15, 0.15]),
    'Kategori': np.random.choice(categories, size=n_samples),
    'Penjualan_Juta': np.random.uniform(5.0, 50.0, size=n_samples),
    'Kepuasan_Pelanggan': np.random.randint(1, 6, size=n_samples)
}

df_sales = pd.DataFrame(data).sort_values('Tanggal').reset_index(drop=True)
print(f"✅ Dataset Penjualan berhasil dibuat ({len(df_sales)} transaksi).")
df_sales.head()

✅ Dataset Penjualan berhasil dibuat (300 transaksi).


,Tanggal,Wilayah,Kategori,Penjualan_Juta,Kepuasan_Pelanggan
0,2026-01-02,Medan,Makanan & Minuman,28.060595,3
1,2026-01-02,Bengkulu,Makanan & Minuman,30.791788,5
2,2026-01-02,Medan,Elektronik,16.424985,3
3,2026-01-04,Bengkulu,Makanan & Minuman,43.088353,4
4,2026-01-05,Surabaya,Elektronik,19.946589,2


In [ ]:
# Widget Interaktif untuk Filtering (Filter on Demand)
wilayah_dropdown = widgets.Dropdown(
    options=['Semua Wilayah'] + list(df_sales['Wilayah'].unique()),
    value='Semua Wilayah',
    description='Wilayah:',
    disabled=False
)

out = widgets.Output()

def update_dashboard(change):
    out.clear_output()
    selected_region = change['new'] if isinstance(change, dict) else wilayah_dropdown.value

    # Filtering Data
    if selected_region == 'Semua Wilayah':
        filtered_df = df_sales
    else:
        filtered_df = df_sales[df_sales['Wilayah'] == selected_region]

    with out:
        # 1. Overview: Tren Penjualan
        fig_line = px.line(
            filtered_df.groupby('Tanggal')['Penjualan_Juta'].sum().reset_index(),
            x='Tanggal', y='Penjualan_Juta',
            title=f'📊 Tren Penjualan Harian ({selected_region})',
            labels={'Penjualan_Juta': 'Total Penjualan (Juta Rp)'},
            template='plotly_white'
        )
        fig_line.update_traces(line_color='#2b5c8f', line_width=2.5)
        fig_line.show()

        # 2. Comparison: Penjualan per Kategori (Bar Chart)
        cat_df = filtered_df.groupby('Kategori')['Penjualan_Juta'].sum().reset_index()
        fig_bar = px.bar(
            cat_df, x='Kategori', y='Penjualan_Juta',
            color='Kategori',
            title='📦 Total Penjualan per Kategori Produk',
            template='plotly_white'
        )
        fig_bar.show()

wilayah_dropdown.observe(update_dashboard, names='value')

# Tampilkan Widget & Dashboard
display(wilayah_dropdown)
display(out)
update_dashboard(None)

Dropdown(description='Wilayah:', options=('Semua Wilayah', 'Medan', 'Bengkulu', 'Surabaya', 'Jakarta'), value=…

Output()

In [ ]:
# Simulasi Data Respon Kuesioner SUS dari 10 Responden (Skala Likert 1-5)
# 10 Pertanyaan Standar SUS (Alternating Positive & Negative Questions)
np.random.seed(10)
sus_responses = np.random.randint(3, 6, size=(10, 10))  # Respon cenderung positif (3-5)

def calculate_sus_score(responses):
    scores = []
    for row in responses:
        score_row = []
        for i, val in enumerate(row):
            if i % 2 == 0:  # Pertanyaan ganjil (1, 3, 5, 7, 9) -> Score = Val - 1
                score_row.append(val - 1)
            else:           # Pertanyaan genap (2, 4, 6, 8, 10) -> Score = 5 - Val
                score_row.append(5 - val)
        total_score = sum(score_row) * 2.5
        scores.append(total_score)
    return scores

sus_scores = calculate_sus_score(sus_responses)
mean_sus = np.mean(sus_scores)

print("📊 --- EVALUASI USABILITAS ANTARMUKA (SUS ANALYSIS) ---")
print(f"Skor SUS Rata-rata: {mean_sus:.2f} / 100")

# Klasifikasi Kategori SUS
if mean_sus >= 80.3:
    grade = "A (Excellent / Highly Usable)"
elif mean_sus >= 68.0:
    grade = "B/C (Good / Above Average)"
else:
    grade = "D/F (Poor / Needs Improvement)"

print(f"Kategori Ketergunaan: {grade}")

📊 --- EVALUASI USABILITAS ANTARMUKA (SUS ANALYSIS) ---
Skor SUS Rata-rata: 48.00 / 100
Kategori Ketergunaan: D/F (Poor / Needs Improvement)


In [ ]:
def generate_ux_recommendations(sus_score):
    print("🎨 --- REKOMENDASI PERBAIKAN DESAIN INTERAKSI (HCI) ---")
    if sus_score >= 80:
        print("1. Visibility of System Status: Antarmuka sangat baik. Pertahankan respon real-time saat filtering data.")
        print("2. Aesthetic and Minimalist Design: Hindari menambah ornamen grafik baru agar tetap bersih (memaksimalkan Data-Ink Ratio).")
    else:
        print("1. Consistency and Standards: Pastikan warna tombol filter dan grafik memiliki konsistensi secara global.")
        print("2. Error Prevention: Sediakan konfirmasi atau tombol 'Reset Filter' jika data tidak ditemukan.")
        print("3. Recognition rather than Recall: Sediakan petunjuk berupa tooltip/hover pada setiap grafik.")

generate_ux_recommendations(mean_sus)

🎨 --- REKOMENDASI PERBAIKAN DESAIN INTERAKSI (HCI) ---
1. Consistency and Standards: Pastikan warna tombol filter dan grafik memiliki konsistensi secara global.
2. Error Prevention: Sediakan konfirmasi atau tombol 'Reset Filter' jika data tidak ditemukan.
3. Recognition rather than Recall: Sediakan petunjuk berupa tooltip/hover pada setiap grafik.
